# Chapter 31 Walkthrough: Sequence Modeling — RNNs, LSTMs, GRUs & Epidemiological Time-Series Forecasting

**Book Reference:** Chapter 31, Sections 31.1 – 31.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. Every forecasting experiment uses the bundled `epi_surveillance` Bangladesh weekly dengue surveillance dataset.

In [ ]:
# @title Step 0: Click Play (▶) to Load Packages & epi_surveillance
!pip install -q sharukh-tutor
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
import sharukh_tutor as tutor

torch.set_num_threads(1)
epi_raw = tutor.load_data("epi_surveillance", as_pandas=True)
dhaka = (
    epi_raw[epi_raw["district"] == "Dhaka"]
    .sort_values(["year", "epi_week"])
    .reset_index(drop=True)
)
n24 = int((dhaka["year"] == 2024).sum())
n25 = int((dhaka["year"] == 2025).sum())
print(f"Dataset shape: {epi_raw.shape} | Dhaka weeks: {len(dhaka)}")
print(f"2024 weeks: {n24} | 2025 weeks: {n25}")
print(
    f"Dhaka dengue cases -> min={dhaka['dengue_cases'].min()}, "
    f"median={dhaka['dengue_cases'].median():.1f}, "
    f"max={dhaka['dengue_cases'].max()}"
)


## Section 31.3.2 — Leak-Free Temporal Splitting & Sliding Windows

In [ ]:
feat_cols = ["dengue_cases", "rainfall_mm", "mean_temp_c"]
raw_vals = dhaka[feat_cols].to_numpy(dtype=np.float32)
n_train = int((dhaka["year"] < 2025).sum())

# Fit StandardScaler strictly on 2024 training rows (zero future leak!)
scaler = StandardScaler()
scaler.fit(raw_vals[:n_train])
scaled = scaler.transform(raw_vals).astype(np.float32)

seq_len = 8
X_tr = np.stack(
    [scaled[t - seq_len : t] for t in range(seq_len, n_train)]
)
y_tr = scaled[seq_len:n_train, 0]
X_te = np.stack(
    [scaled[t - seq_len : t] for t in range(n_train, len(dhaka))]
)
y_te = scaled[n_train:, 0]

print(f"Train scaler mean (2024 only): {np.round(scaler.mean_, 2)}")
print(f"Full-series mean (LEAKY!):     {np.round(raw_vals.mean(0), 2)}")
print(f"X_train: {X_tr.shape}, y_train: {y_tr.shape}")
print(f"X_test:  {X_te.shape}, y_test:  {y_te.shape}")


## Section 31.3.3 — Pure-NumPy Elman RNN Cell & Vanishing/Exploding Gradient Demo

In [ ]:
rng = np.random.default_rng(42)
x_t = np.array([[0.5, -0.2, 1.1], [-0.4, 0.8, 0.3]])
h_prev = np.zeros((2, 4))
W_xh = rng.normal(0.0, 0.3, size=(3, 4))
W_hh = rng.normal(0.0, 0.3, size=(4, 4))
b_h = np.zeros(4)

h_next = np.tanh(x_t @ W_xh + h_prev @ W_hh + b_h)
print(f"RNN step output shape: {h_next.shape}")
print(f"h_next[0]: {np.round(h_next[0], 4)}")

# Why vanilla RNN gradients vanish or explode over long horizons:
for steps in [5, 10, 20, 30]:
    print(
        f"After {steps:2d} steps -> "
        f"0.5^{steps:2d} = {0.5 ** steps:.2e} | "
        f"1.5^{steps:2d} = {1.5 ** steps:.2e}"
    )


## Section 31.3.4 — PyTorch `nn.GRU` & `nn.LSTM` `batch_first=True` Tensor Shapes

In [ ]:
torch.manual_seed(42)
sample_x = torch.tensor(X_tr[:4], dtype=torch.float32)

gru = nn.GRU(input_size=3, hidden_size=16, batch_first=True)
gru_out, gru_hn = gru(sample_x)

lstm = nn.LSTM(input_size=3, hidden_size=16, batch_first=True)
lstm_out, (lstm_hn, lstm_cn) = lstm(sample_x)

print(f"Input x:  {tuple(sample_x.shape)}  (B, T, F)")
print(f"GRU out:  {tuple(gru_out.shape)} | h_n: {tuple(gru_hn.shape)}")
print(
    f"LSTM out: {tuple(lstm_out.shape)} | "
    f"h_n: {tuple(lstm_hn.shape)} | c_n: {tuple(lstm_cn.shape)}"
)


## Section 31.3.5 — Variable-Length Patient Vitals: Padding & Boolean Masking

In [ ]:
vitals = [
    torch.tensor([[98.0, 120.0], [97.5, 118.0], [96.0, 115.0]]),
    torch.tensor([
        [99.0, 110.0], [98.5, 112.0], [98.0, 114.0],
        [97.0, 116.0], [96.5, 119.0],
    ]),
    torch.tensor([[95.0, 130.0], [94.0, 135.0]]),
]
lengths = torch.tensor([s.shape[0] for s in vitals], dtype=torch.long)
padded = nn.utils.rnn.pad_sequence(
    vitals, batch_first=True, padding_value=0.0
)
max_len = padded.shape[1]
valid_mask = torch.arange(max_len).unsqueeze(0) < lengths.unsqueeze(1)

print(f"Padded shape: {tuple(padded.shape)} | lengths: {lengths.tolist()}")
print("Valid mask:", valid_mask.int().tolist())


## Section 31.3.6 — Persistence vs. GRU vs. LSTM on Held-Out 2025 Dhaka Dengue Weeks

In [ ]:
class EpiGRUForecaster(nn.Module):
    def __init__(self, input_size, hidden_size=16):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out[:, -1, :])

class EpiLSTMForecaster(nn.Module):
    def __init__(self, input_size, hidden_size=16):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])

def fit_forecaster(
    model_cls, X_train, y_train, X_test,
    hidden_size=16, epochs=40, lr=0.02, seed=42,
):
    torch.manual_seed(seed)
    model = model_cls(X_train.shape[-1], hidden_size=hidden_size)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    xt = torch.tensor(X_train, dtype=torch.float32)
    yt = torch.tensor(y_train, dtype=torch.float32).unsqueeze(-1)
    xe = torch.tensor(X_test, dtype=torch.float32)
    model.train()
    for _ in range(epochs):
        opt.zero_grad()
        loss = loss_fn(model(xt), yt)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    model.eval()
    with torch.no_grad():
        preds = model(xe).squeeze(-1).numpy()
    return preds, float(loss.item())

pers_preds = X_te[:, -1, 0]
gru_preds, _ = fit_forecaster(EpiGRUForecaster, X_tr, y_tr, X_te)
lstm_preds, _ = fit_forecaster(EpiLSTMForecaster, X_tr, y_tr, X_te)

rows = []
for name, p in [
    ("Persistence", pers_preds),
    ("GRU", gru_preds),
    ("LSTM", lstm_preds),
]:
    rmse_z = float(np.sqrt(np.mean((y_te - p) ** 2)))
    mae_z = float(np.mean(np.abs(y_te - p)))
    rows.append({
        "model": name,
        "rmse_z": round(rmse_z, 4),
        "mae_z": round(mae_z, 4),
        "rmse_cases": round(rmse_z * float(scaler.scale_[0]), 2),
    })
lb = pd.DataFrame(rows).sort_values("rmse_z").reset_index(drop=True)
print(lb.to_string(index=False))


## Section 31.3.7 — Break It On Purpose! (Sequence Modeling Error Anatomy)

In [ ]:
# Intentional Bug #1: 2D (B, F) input instead of 3D (B, T, F) into nn.GRU
bad_x_2d = torch.zeros(4, 3)
h0_batched = torch.zeros(1, 4, 16)
nn.GRU(3, 16, batch_first=True)(bad_x_2d, h0_batched)


In [ ]:
# Intentional Bug #2: Mismatched feature count (5 vs expected input_size=3)
bad_x_feat = torch.zeros(4, 8, 5)
nn.LSTM(input_size=3, hidden_size=16, batch_first=True)(bad_x_feat)


In [ ]:
# Intentional Bug #3: Unpacking nn.LSTM output into 3 flat variables
x_ok = torch.zeros(4, 8, 3)
out, h_n, c_n = nn.LSTM(3, 16, batch_first=True)(x_ok)


## Section 31.4 — Stage 4 Blueprint: Multi-District Dengue Forecasting Benchmark

In [ ]:
def forecast_district_suite(
    epi_df,
    districts=("Dhaka", "Chattogram", "Sylhet"),
    seq_len=8,
    cutoff_year=2025,
    seed=42,
):
    """Benchmark Persistence, GRU & LSTM across Bangladesh districts."""
    rows = []
    cols = ["dengue_cases", "rainfall_mm", "mean_temp_c"]
    for dist in districts:
        sub = (
            epi_df[epi_df["district"] == dist]
            .sort_values(["year", "epi_week"])
            .reset_index(drop=True)
        )
        raw = sub[cols].to_numpy(dtype=np.float32)
        n_tr = int((sub["year"] < cutoff_year).sum())
        sc = StandardScaler().fit(raw[:n_tr])
        z = sc.transform(raw).astype(np.float32)
        X_tr = np.stack(
            [z[t - seq_len : t] for t in range(seq_len, n_tr)]
        )
        y_tr = z[seq_len:n_tr, 0]
        X_te = np.stack(
            [z[t - seq_len : t] for t in range(n_tr, len(sub))]
        )
        y_te = z[n_tr:, 0]
        p_pers = X_te[:, -1, 0]
        p_gru, _ = fit_forecaster(
            EpiGRUForecaster, X_tr, y_tr, X_te, seed=seed
        )
        p_lstm, _ = fit_forecaster(
            EpiLSTMForecaster, X_tr, y_tr, X_te, seed=seed
        )
        rmses = {
            "Persistence": float(np.sqrt(np.mean((y_te - p_pers) ** 2))),
            "GRU": float(np.sqrt(np.mean((y_te - p_gru) ** 2))),
            "LSTM": float(np.sqrt(np.mean((y_te - p_lstm) ** 2))),
        }
        winner = min(rmses, key=rmses.get)
        rows.append({
            "district": dist,
            "pers_rmse": round(rmses["Persistence"], 4),
            "gru_rmse": round(rmses["GRU"], 4),
            "lstm_rmse": round(rmses["LSTM"], 4),
            "winner": winner,
        })
    return pd.DataFrame(rows)

print(forecast_district_suite(epi_raw).to_string(index=False))
